# Flood events in Bangladesh (NASA EONET)

Run the code cell below to download the flood events from the [NASA EONET API](https://eonet.gsfc.nasa.gov/api/v3/events/geojson?category=floods&status=all&limit=500), filter them against Bangladesh's country boundary, and display an interactive map plus a table of matching event names, dates, and links.

**How to read this:** the map shows EONET-reported event geometries that overlap Bangladesh's boundary; a matching event is not independent confirmation that every point inside its geometry was flooded. Names are the labels supplied by EONET and may not identify a village or upazila. The request asks for at most 500 records, so this is not a complete historical archive if more events exist beyond that limit. An internet connection is required.


In [5]:
import pandas as pd
import folium
import requests
from IPython.display import display
from shapely.geometry import shape

EONET_URL = "https://eonet.gsfc.nasa.gov/api/v3/events/geojson?category=floods&status=all"
GEOBND_URL = "https://www.geoboundaries.org/api/current/gbOpen/BGD/ADM0/"

# Download flood events and Bangladesh's official country outline.
eonet_response = requests.get(EONET_URL, timeout=30)
eonet_response.raise_for_status()
eonet_features = eonet_response.json().get("features", [])

boundary_metadata_response = requests.get(GEOBND_URL, timeout=30)
boundary_metadata_response.raise_for_status()
boundary_url = boundary_metadata_response.json()["gjDownloadURL"]
boundary_response = requests.get(boundary_url, timeout=30)
boundary_response.raise_for_status()
boundary_geojson = boundary_response.json()
bangladesh_boundary = shape(boundary_geojson["features"][0]["geometry"])

# Keep events whose reported geometry overlaps Bangladesh.
bangladesh_events = []
event_rows = []
for feature in eonet_features:
    geometry = feature.get("geometry")
    if not geometry:
        continue

    if not shape(geometry).intersects(bangladesh_boundary):
        continue

    properties = feature.get("properties") or {}
    title = properties.get("title") or "Flood event (title unavailable)"
    event_date = properties.get("date") or properties.get("closed") or "Unknown"
    event_link = properties.get("link", "")

    map_feature = {
        "type": "Feature",
        "geometry": geometry,
        "properties": {
            "_event_title": title,
            "_event_date": event_date,
        },
    }
    bangladesh_events.append(map_feature)
    event_rows.append({
        "Place / event reported by NASA EONET": title,
        "Date (UTC)": event_date,
        "EONET event link": event_link,
    })

# Draw the country outline and all matching EONET event geometries.
center = bangladesh_boundary.representative_point()
flood_map = folium.Map(location=[center.y, center.x], zoom_start=7, tiles="OpenStreetMap")
folium.GeoJson(
    boundary_geojson,
    name="Bangladesh boundary",
    style_function=lambda _: {"color": "#166534", "weight": 2, "fillOpacity": 0.03},
).add_to(flood_map)

if bangladesh_events:
    folium.GeoJson(
        {"type": "FeatureCollection", "features": bangladesh_events},
        name="Flood events",
        style_function=lambda _: {"color": "#dc2626", "weight": 2, "fillColor": "#ef4444", "fillOpacity": 0.35},
        marker=folium.CircleMarker(radius=7, color="#b91c1c", fill=True, fill_color="#ef4444", fill_opacity=0.85),
        tooltip=folium.GeoJsonTooltip(
            fields=["_event_title", "_event_date"],
            aliases=["Event:", "Date (UTC):"],
            sticky=True,
        ),
    ).add_to(flood_map)

folium.LayerControl().add_to(flood_map)
min_lon, min_lat, max_lon, max_lat = bangladesh_boundary.bounds
flood_map.fit_bounds([[min_lat, min_lon], [max_lat, max_lon]])

events_df = pd.DataFrame(
    event_rows,
    columns=["Place / event reported by NASA EONET", "Date (UTC)", "EONET event link"],
)
if not events_df.empty:
    events_df = events_df.sort_values("Date (UTC)", ascending=False, ignore_index=True)

print(f"Bangladesh matches: {len(events_df)} of {len(eonet_features)} returned flood events")
if len(eonet_features) >= 500:
    print("Note: The API returned the requested 500-event limit; older events outside this response are not included.")
if events_df.empty:
    print("No returned EONET flood geometry intersects Bangladesh.")
display(events_df)
flood_map


Bangladesh matches: 9 of 992 returned flood events
Note: The API returned the requested 500-event limit; older events outside this response are not included.


,Place / event reported by NASA EONET,Date (UTC),EONET event link
0,Flood in Bangladesh 1103878,2026-05-12T20:00:00Z,https://eonet.gsfc.nasa.gov/api/v3/events/EONE...
1,Flood in Bangladesh 1103853,2026-04-27T20:00:00Z,https://eonet.gsfc.nasa.gov/api/v3/events/EONE...
2,Flood in Bangladesh 1103509,2025-09-17T20:00:00Z,https://eonet.gsfc.nasa.gov/api/v3/events/EONE...
3,Flood in Bangladesh 1103308,2025-05-29T20:00:00Z,https://eonet.gsfc.nasa.gov/api/v3/events/EONE...
4,"Nepal, India, and Bangladesh Monsoon Floods",2017-08-12T00:00:00Z,https://eonet.gsfc.nasa.gov/api/v3/events/EONE...
5,Bangladesh Floods and Landslides,2017-05-30T00:00:00Z,https://eonet.gsfc.nasa.gov/api/v3/events/EONE...
6,"Bangladesh Flooding, May 2016",2016-05-21T00:00:00Z,https://eonet.gsfc.nasa.gov/api/v3/events/EONE...
7,"Northeast India Floods, August 2015",2015-08-01T00:00:00Z,https://eonet.gsfc.nasa.gov/api/v3/events/EONE...
8,"Bangladesh: Floods and Landslides, Summer 2015",2015-06-24T00:00:00Z,https://eonet.gsfc.nasa.gov/api/v3/events/EONE...


In [6]:
from datetime import date
import json
import requests
import pandas as pd
import folium
from IPython.display import display

# Bangladesh bounding box:
# minimum longitude, maximum latitude, maximum longitude, minimum latitude
BBOX = "88,27,93,20"

API_URL = "https://eonet.gsfc.nasa.gov/api/v3/events/geojson"
all_features = []

# EONET-এ বছর ধরে খুঁজি, যাতে শুধু সাম্প্রতিক ইভেন্টে সীমাবদ্ধ না থাকি।
first_year = 2000
last_year = date.today().year

for year in range(first_year, last_year + 1):
    start_date = f"{year}-01-01"
    end_date = f"{year}-12-31"

    if year == last_year:
        end_date = date.today().isoformat()

    params = {
        "category": "floods",
        "status": "all",
        "start": start_date,
        "end": end_date,
        "bbox": BBOX,
        "limit": 500,
    }

    response = requests.get(API_URL, params=params, timeout=60)
    response.raise_for_status()
    result = response.json()

    year_features = result.get("features", [])
    all_features.extend(year_features)

    print(f"{year}: {len(year_features)}টি রেকর্ড")


# একই event/geometry একাধিকবার এলে বাদ দিই।
unique_features = []
seen = set()

for feature in all_features:
    props = feature.get("properties", {})
    geometry = feature.get("geometry")
    key = (
        props.get("id"),
        props.get("date"),
        json.dumps(geometry, sort_keys=True),
    )

    if key not in seen:
        seen.add(key)
        unique_features.append(feature)


def get_positions(coords):
    """GeoJSON coordinates থেকে [longitude, latitude] জোড়া বের করে।"""
    positions = []

    def walk(value):
        if (
            isinstance(value, list)
            and len(value) >= 2
            and isinstance(value[0], (int, float))
            and isinstance(value[1], (int, float))
        ):
            positions.append((float(value[0]), float(value[1])))
        elif isinstance(value, list):
            for item in value:
                walk(item)

    walk(coords)
    return positions


# টেবিলের জন্য প্রতিটি ইভেন্টের অবস্থানের একটি আনুমানিক কেন্দ্র বানাই।
rows = []

for feature in unique_features:
    props = feature.get("properties", {})
    geometry = feature.get("geometry") or {}
    positions = get_positions(geometry.get("coordinates", []))

    if positions:
        center_lon = sum(lon for lon, lat in positions) / len(positions)
        center_lat = sum(lat for lon, lat in positions) / len(positions)
    else:
        center_lon = None
        center_lat = None

    sources = props.get("sources", [])
    source_names = ", ".join(
        source.get("id", "") for source in sources if isinstance(source, dict)
    )

    rows.append({
        "Date": props.get("date", ""),
        "Event": props.get("title", ""),
        "Event ID": props.get("id", ""),
        "Geometry type": geometry.get("type", ""),
        "Approx latitude": center_lat,
        "Approx longitude": center_lon,
        "Sources": source_names,
    })

df = pd.DataFrame(rows)

if not df.empty:
    df = df.sort_values(["Date", "Event"], na_position="last").reset_index(drop=True)

print(f"\nমোট পাওয়া বন্যা ইভেন্ট: {len(df)}")
display(df)


# মানচিত্রে ঘটনার জ্যামিতি দেখাই।
flood_map = folium.Map(
    location=[23.7, 90.4],
    zoom_start=6,
    tiles="OpenStreetMap",
)

if unique_features:
    event_geojson = {
        "type": "FeatureCollection",
        "features": unique_features,
    }

    folium.GeoJson(
        event_geojson,
        name="EONET flood events",
        tooltip=folium.GeoJsonTooltip(
            fields=["title", "date", "id"],
            aliases=["Event:", "Date:", "Event ID:"],
            localize=True,
            sticky=True,
        ),
        style_function=lambda feature: {
            "color": "#d62828",
            "weight": 3,
            "fillColor": "#f77f00",
            "fillOpacity": 0.25,
        },
        marker=folium.CircleMarker(
            radius=6,
            color="#d62828",
            fill=True,
            fill_color="#f77f00",
            fill_opacity=0.8,
        ),
    ).add_to(flood_map)

    folium.LayerControl().add_to(flood_map)

# Bangladesh-এর আনুমানিক সীমানায় map view রাখি।
flood_map.fit_bounds([[20.0, 88.0], [27.0, 93.0]])

display(flood_map)


# ফলাফল ফাইল হিসেবে সংরক্ষণ করি।
df.to_csv("bangladesh_eonet_flood_events.csv", index=False, encoding="utf-8-sig")

with open("bangladesh_eonet_flood_events.geojson", "w", encoding="utf-8") as f:
    json.dump(
        {"type": "FeatureCollection", "features": unique_features},
        f,
        ensure_ascii=False,
        indent=2,
    )

print(
    "\nফাইল তৈরি হয়েছে:\n"
    "- bangladesh_eonet_flood_events.csv\n"
    "- bangladesh_eonet_flood_events.geojson"
)

2000: 0টি রেকর্ড
2001: 0টি রেকর্ড
2002: 0টি রেকর্ড
2003: 0টি রেকর্ড
2004: 0টি রেকর্ড
2005: 0টি রেকর্ড
2006: 0টি রেকর্ড
2007: 0টি রেকর্ড
2008: 0টি রেকর্ড
2009: 0টি রেকর্ড
2010: 0টি রেকর্ড
2011: 0টি রেকর্ড
2012: 0টি রেকর্ড
2013: 0টি রেকর্ড
2014: 0টি রেকর্ড
2015: 1টি রেকর্ড
2016: 1টি রেকর্ড
2017: 1টি রেকর্ড
2018: 0টি রেকর্ড
2019: 0টি রেকর্ড
2020: 0টি রেকর্ড
2021: 0টি রেকর্ড
2022: 0টি রেকর্ড
2023: 0টি রেকর্ড
2024: 0টি রেকর্ড
2025: 2টি রেকর্ড
2026: 7টি রেকর্ড

মোট পাওয়া বন্যা ইভেন্ট: 12


,Date,Event,Event ID,Geometry type,Approx latitude,Approx longitude,Sources
0,2015-06-24T00:00:00Z,"Bangladesh: Floods and Landslides, Summer 2015",EONET_42,Polygon,21.997501,91.890625,ReliefWeb
1,2016-05-21T00:00:00Z,"Bangladesh Flooding, May 2016",EONET_415,Polygon,22.552550,90.154297,"IDC, ReliefWeb"
2,2017-05-30T00:00:00Z,Bangladesh Floods and Landslides,EONET_2835,Polygon,21.911756,91.806641,"GDACS, IDC, ReliefWeb"
3,2025-05-29T20:00:00Z,Flood in Bangladesh 1103308,EONET_13855,Point,23.107800,91.988996,GDACS
4,2025-09-17T20:00:00Z,Flood in Bangladesh 1103509,EONET_15532,Point,25.093485,90.097825,GDACS
5,2026-04-18T20:00:00Z,Flood in India 1103844,EONET_19811,Point,26.160045,91.695809,GDACS
6,2026-04-27T20:00:00Z,Flood in Bangladesh 1103853,EONET_19926,Point,24.488082,91.763018,GDACS
7,2026-05-12T20:00:00Z,Flood in Bangladesh 1103878,EONET_20104,Point,25.122257,90.339562,GDACS
8,2026-07-03T20:00:00Z,Flood in Myanmar 1104011,EONET_21253,Polygon,92.947147,20.881378,GDACS
9,2026-07-08T20:00:00Z,Flood in Bangladesh 1104015,EONET_21272,Polygon,91.419263,24.359669,GDACS



ফাইল তৈরি হয়েছে:
- bangladesh_eonet_flood_events.csv
- bangladesh_eonet_flood_events.geojson


In [7]:
import ee
import geemap
from datetime import datetime, timezone, timedelta

# প্রথমবার চালালে Google account দিয়ে sign in করতে বলবে।
ee.Authenticate()

# nasaproject-513305 বদলে নিজের Google Cloud project ID দিন।
ee.Initialize(project="nasaproject-513305")


# বাংলাদেশের আনুমানিক সীমানা
bangladesh = ee.Geometry.Rectangle([88.0, 20.0, 93.0, 27.0])

# DSWx-S1 উৎপাদন শুরু: 2024 সালের সেপ্টেম্বর।
# Earth Engine filterDate-এ শেষের তারিখটি বাদ পড়ে, তাই আগামীকাল ব্যবহার করছি।
start_date = "2022-09-01"
end_date = (
    datetime.now(timezone.utc).date() + timedelta(days=1)
).isoformat()

# NASA OPERA DSWx-S1 image collection
collection = (
    ee.ImageCollection("OPERA/DSWX/L3_V1/S1")
    .filterBounds(bangladesh)
    .filterDate(start_date, end_date)
    .select("BWTR_Binary_water")
)

print("সময়কাল:", start_date, "থেকে", end_date)
print("DSWx-S1 scenes:", collection.size().getInfo())


def prepare_observation(image):
    """
    BWTR মানে:
    0 = water নয়
    1 = water (open water বা inundated vegetation)
    250+ = masked/invalid; এগুলো বাদ
    """
    bwtr = image.select("BWTR_Binary_water")
    valid = bwtr.lt(250)

    water_observation = (
        bwtr.eq(1)
        .updateMask(valid)
        .unmask(0)
        .rename("water_observations")
    )

    valid_observation = (
        valid
        .unmask(0)
        .rename("valid_observations")
    )

    return (
        water_observation
        .addBands(valid_observation)
        .copyProperties(image, ["system:time_start"])
    )


# Pixel-ভিত্তিক water detection ও valid observation-এর нийт সংখ্যা
summary = collection.map(prepare_observation).sum().clip(bangladesh)

water_count = summary.select("water_observations")
valid_count = summary.select("valid_observations")

# 90% বা তার বেশি সময় জল হিসেবে দেখা জায়গাকে সম্ভাব্য স্থায়ী জল ধরে
# মূল ফলাফলের flood-signal layer থেকে বাদ দিই।
# এই 90% একটি বিশ্লেষণী অনুমান—NASA-র নির্ধারিত flood threshold নয়।
jrc_water_occurrence = (
    ee.Image("JRC/GSW1_4/GlobalSurfaceWater")
    .select("occurrence")
    .unmask(0)
)

likely_permanent_water = jrc_water_occurrence.gte(90)
not_likely_permanent_water = likely_permanent_water.Not()

# প্রতিটি pixel-এ যতগুলো DSWx observation-এ জল পাওয়া গেছে
possible_inundation_count = (
    water_count
    .updateMask(not_likely_permanent_water)
    .rename("possible_inundation_observation_count")
)

# valid observations-এর মধ্যে কত অংশে জল দেখা গেছে: 0 থেকে 1
possible_inundation_frequency = (
    water_count
    .divide(valid_count)
    .updateMask(valid_count.gt(0))
    .updateMask(not_likely_permanent_water)
    .rename("possible_inundation_frequency")
)

# অন্তত দুইটি আলাদা satellite observation-এ জল ধরা পড়া pixel
repeated_detection = (
    possible_inundation_count
    .updateMask(possible_inundation_count.gte(2))
    .rename("water_detected_at_least_twice")
)


# JupyterLab-এ ইন্টার‌্যাক্টিভ ম্যাপ
Map = geemap.Map()
Map.centerObject(bangladesh, 6)

Map.addLayer(
    possible_inundation_frequency,
    {
        "min": 0,
        "max": 0.5,
        "palette": ["fff7bc", "fec44f", "d95f0e", "7f0000"],
    },
    "সম্ভাব্য মৌসুমি জল: valid observation-এর অনুপাত",
)

Map.addLayer(
    possible_inundation_count,
    {
        "min": 1,
        "max": 10,
        "palette": ["deebf7", "9ecae1", "3182bd", "08306b"],
    },
    "স্থায়ী জল বাদে জল শনাক্ত হওয়ার observation সংখ্যা",
)

Map.addLayer(
    repeated_detection,
    {
        "min": 2,
        "max": 10,
        "palette": ["fee8c8", "fdbb84", "e34a33", "b30000"],
    },
    "অন্তত ২টি observation-এ জল শনাক্ত",
)

Map.addLayer(
    likely_permanent_water.selfMask().clip(bangladesh),
    {"palette": ["2864dc"]},
    "সম্ভাব্য স্থায়ী জল (JRC occurrence ≥ 90%)",
)

Map

সময়কাল: 2022-09-01 থেকে 2026-11-07
DSWx-S1 scenes: 16906


Map(center=[23.48880156808697, 90.49999999999989], controls=(WidgetControl(options=['position', 'transparent_b…

In [8]:
from IPython.display import clear_output
from ipywidgets import Output

_previous_flood_click_handler = globals().get("flood_click_handler")
if _previous_flood_click_handler is not None:
    Map.on_interaction(_previous_flood_click_handler, remove=True)

flood_click_output = Output(layout={"border": "1px solid #999", "padding": "8px"})


def check_flood_prone(latitude: float, longitude: float) -> None:
    """Print flood-prone status from non-permanent-water observations.

    At least two water detections in valid OPERA DSWx-S1 observations since
    2022-09-01 indicate potential flood-proneness; this is not an official
    flood-risk prediction.
    """
    if not -90 <= latitude <= 90 or not -180 <= longitude <= 180:
        raise ValueError("Latitude or longitude is outside its valid range.")

    point = ee.Geometry.Point([longitude, latitude])
    if not bangladesh.contains(point).getInfo():
        print("Selected point is outside the Bangladesh analysis area.")
        return

    values = ee.Image.cat([
        jrc_water_occurrence.rename("permanent_water_occurrence"),
        water_count.rename("water_detections"),
        valid_count.rename("valid_observations"),
    ]).reduceRegion(
        reducer=ee.Reducer.first(),
        geometry=point,
        scale=30,
        maxPixels=1_000_000,
    ).getInfo()

    permanent_occurrence = values.get("permanent_water_occurrence")
    detections = values.get("water_detections")
    observations = values.get("valid_observations")

    print(f"Selected location: {latitude:.5f}, {longitude:.5f}")
    if permanent_occurrence is None:
        print("Flood-prone: INSUFFICIENT DATA (permanent-water data is unavailable).")
        return
    if permanent_occurrence >= 90:
        print("Flood-prone: NOT CLASSIFIED (permanent water is excluded).")
        return
    if observations is None or observations < 2 or detections is None:
        print("Flood-prone: INSUFFICIENT DATA (fewer than 2 valid observations).")
        return

    detections = detections or 0
    frequency = detections / observations
    if detections >= 2:
        print(
            "Flood-prone: YES (non-permanent water detected in "
            f"{int(detections)} of {int(observations)} valid observations; "
            f"frequency {frequency:.1%})."
        )
    else:
        print(
            "Flood-prone: NO (non-permanent water detected in fewer than "
            f"2 observations; {int(detections)} of {int(observations)} valid)."
        )


def report_clicked_location(**kwargs) -> None:
    if kwargs.get("type") == "click":
        latitude, longitude = kwargs["coordinates"]
        with flood_click_output:
            clear_output(wait=True)
            print(f"Checking selected location: {latitude:.5f}, {longitude:.5f} ...")
            check_flood_prone(latitude, longitude)


flood_click_handler = report_clicked_location
Map.on_interaction(flood_click_handler)
print("Click a location on the map. The classification will appear below:")
display(flood_click_output)
display(Map)


Click a location on the map. The classification will appear below:


Output(layout=Layout(border_bottom='1px solid #999', border_left='1px solid #999', border_right='1px solid #99…

Map(center=[23.48880156808697, 90.49999999999989], controls=(WidgetControl(options=['position', 'transparent_b…